# E-Commerce Hypothesis Testing & Correlation Analysis

**Project Overview:**  
This notebook applies inferential statistics—including Welch's two-sample t-tests, Chi-square tests of independence, and Pearson correlation analyses—to an e-commerce dataset (`transactions.csv`, `customers.csv`, and `web_sessions.csv`) developed as a data analysis course project.

**Methodology & Reporting Standard:**  
For every business hypothesis and analytical challenge, we:
1. Execute the appropriate statistical test (`scipy.stats`).
2. Report exact test statistics and $p$-values (evaluated at $\alpha = 0.05$).
3. Translate the statistical outcome into plain English.
4. Evaluate practical business significance and strategic implications.

## 1. Data Loading, Merging & Granularity Check

In [1]:
import pandas as pd
from scipy import stats

# Load datasets using relative paths
transactions = pd.read_csv("data/transactions.csv")
customers = pd.read_csv("data/customers.csv")
web = pd.read_csv("data/web_sessions.csv")

# Create analytical views at appropriate granularities
tx_cust = transactions.merge(customers, on="customer_id", how="left")
tx_web = transactions.merge(web, on="customer_id", how="left")
full = tx_cust.merge(web, on="customer_id", how="left")

# Verify row counts before and after merges
print("Base table row counts:")
print(f"  transactions: {len(transactions):,}")
print(f"  customers:    {len(customers):,}")
print(f"  web_sessions: {len(web):,}")

print("\nMerged table row counts:")
print(f"  tx_cust (transactions + customers): {len(tx_cust):,}")
print(f"  tx_web  (transactions + web):       {len(tx_web):,}")
print(f"  full    (all three tables):         {len(full):,}")

Base table row counts:
  transactions: 45,000
  customers:    10,000
  web_sessions: 69,590

Merged table row counts:
  tx_cust (transactions + customers): 45,000
  tx_web  (transactions + web):       313,313
  full    (all three tables):         313,313

**Methodological Note on Merge Granularity & Row Duplication:**  
 - `tx_cust` preserves the exact `45,000` transaction rows because `customer_id` is unique in `customers` (1-to-many join). Therefore, all transaction-value comparisons (Q1–Q4, Q7) use `tx_cust` without row duplication.  
 - Merging `web_sessions` onto `transactions` (`tx_web` and `full`) expands the dataset from `45,000` to `313,313` rows because individual customers have multiple browsing sessions (many-to-many join on `customer_id`).  
 - **How this is handled:** Pure session-level questions (Q5, Q10, C2) are evaluated directly on the un-duplicated `web` table (`69,590` sessions), and customer-level spend (C1) is aggregated by `customer_id` prior to joining `web`. Where cross-table exploration is required (Q6, Q8, Q9), note that repeated customer-session pairs inflate sample size $N$, making tiny effects appear statistically significant—a key theme explored in Challenge C3.

## 2. Hypothesis Testing & Correlation Questions

### Q1. Do customers from the UK spend more per transaction than customers from Germany?

In [2]:
uk = tx_cust[tx_cust["country"] == "UK"]["purchase_value"].dropna()
de = tx_cust[tx_cust["country"] == "Germany"]["purchase_value"].dropna()

t_stat, p_val = stats.ttest_ind(uk, de, equal_var=False)

print(f"UK mean:      ${uk.mean():.2f}")
print(f"Germany mean: ${de.mean():.2f}")
print(f"t-statistic:  {t_stat:.4f}")
print(f"p-value:      {p_val:.4f}")

UK mean:      $234.84
Germany mean: $239.03
t-statistic:  -0.8612
p-value:      0.3892

- **Result:** UK Mean = `$234.84`, Germany Mean = `$239.03` | $t = -0.8612$, $p = 0.3892$
- Because $p = 0.3892 > 0.05$, there is no statistically significant difference in average purchase value between customers in the UK and Germany. The ~$4.19 difference is consistent with random sampling variation.
- **Business Meaning:** Purchasing power per order is comparable across both markets. Pricing tiers and promotional thresholds do not need country-specific adjustments between the UK and Germany.

### Q2. Do discounted purchases differ in average value from non-discounted purchases?

In [3]:
disc = tx_cust[tx_cust["discount_used"] == 1]["purchase_value"].dropna()
nodisc = tx_cust[tx_cust["discount_used"] == 0]["purchase_value"].dropna()

t_stat, p_val = stats.ttest_ind(disc, nodisc, equal_var=False)

print(f"Discounted mean:     ${disc.mean():.2f}")
print(f"Non-discounted mean: ${nodisc.mean():.2f}")
print(f"t-statistic:         {t_stat:.4f}")
print(f"p-value:             {p_val:.4f}")

Discounted mean:     $235.61
Non-discounted mean: $234.56
t-statistic:         0.4013
p-value:             0.6882

- **Result:** Discounted Mean = `$235.61`, Non-Discounted Mean = `$234.56` | $t = 0.4013$, $p = 0.6882$
- With $p = 0.6882 > 0.05$, there is no statistically significant difference in average order value between discounted and full-price transactions.
- **Business Meaning:** Discounts do not encourage customers to build larger baskets, nor do they cannibalize order size. Rather than offering blanket discounts to boost Average Order Value (AOV), the business should use tiered minimum-spend coupons (e.g., *"Spend $300, save $30"*).

### Q3. Do male and female customers differ in average purchase value?

In [4]:
male = tx_cust[tx_cust["gender"] == "Male"]["purchase_value"].dropna()
female = tx_cust[tx_cust["gender"] == "Female"]["purchase_value"].dropna()

t_stat, p_val = stats.ttest_ind(male, female, equal_var=False)

print(f"Male mean:   ${male.mean():.2f}")
print(f"Female mean: ${female.mean():.2f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value:     {p_val:.4f}")

Male mean:   $234.81
Female mean: $235.94
t-statistic: -0.3875
p-value:     0.6984

- **Result:** Male Mean = `$234.81`, Female Mean = `$235.94` | $t = -0.3875$, $p = 0.6984$
- Because $p = 0.6984 > 0.05$, male and female customers spend virtually the same amount per transaction on average.
- **Business Meaning:** Gender is not a meaningful driver of order value. Marketing budget allocation should prioritize behavioral and category-affinity segmentation over gender-based spend targeting.

### Q4. Do customers acquired via email spend more than those from social channels?

In [5]:
email = tx_cust[tx_cust["acquisition_channel"].str.lower() == "email"]["purchase_value"].dropna()
social = tx_cust[tx_cust["acquisition_channel"].str.lower() == "social"]["purchase_value"].dropna()

t_stat, p_val = stats.ttest_ind(email, social, equal_var=False)

print(f"Email mean:  ${email.mean():.2f}")
print(f"Social mean: ${social.mean():.2f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value:     {p_val:.4f}")

Email mean:  $236.82
Social mean: $232.19
t-statistic: 1.2402
p-value:     0.2149

- **Result:** Email Mean = `$236.82`, Social Mean = `$232.19` | $t = 1.2402$, $p = 0.2149$
- Since $p = 0.2149 > 0.05$, the observed $4.63 difference between email-acquired and social-acquired customers is not statistically significant.
- **Business Meaning:** Customers acquired via email and social media exhibit similar per-order spending once converted; channel ROI comparisons should therefore focus on Customer Acquisition Cost (CAC) and conversion volume rather than AOV.

### Q5. Is device type associated with conversion?

In [6]:
device_conv = web[["device_type", "converted"]].dropna()
cont_table = pd.crosstab(device_conv["device_type"], device_conv["converted"])

chi2, p_val, dof, expected = stats.chi2_contingency(cont_table)

print(cont_table)
print(f"\nChi-square: {chi2:.4f}")
print(f"p-value:    {p_val:.4f}")

converted        0     1
device_type             
desktop      20020  3068
mobile       20213  3042
tablet       20244  3013

Chi-square: 1.1502
p-value:    0.5626

- **Result:** $\chi^2 = 1.1502$ ($\text{dof} = 2$), $p = 0.5626$ (Conversion rates: Desktop `13.29%`, Mobile `13.08%`, Tablet `12.96%`)
- With $p = 0.5626 > 0.05$, conversion rates are statistically independent of device type. Users convert at virtually identical rates (~13%) across desktop, mobile, and tablet.
- **Business Meaning:** The e-commerce checkout experience performs consistently across screen sizes, indicating no major mobile UX bottleneck relative to desktop.

### Q6. Is acquisition channel associated with conversion?

In [7]:
# Standardize letter case to merge duplicate labels ("Email" vs "email", "paid_Search" vs "paid_search")
acq_conv = full[["acquisition_channel", "converted"]].dropna().copy()
acq_conv["acquisition_channel"] = acq_conv["acquisition_channel"].str.lower()

cont_table = pd.crosstab(acq_conv["acquisition_channel"], acq_conv["converted"])
chi2, p_val, dof, expected = stats.chi2_contingency(cont_table)

print(cont_table)
print(f"\nChi-square: {chi2:.4f} (dof={dof})")
print(f"p-value:    {p_val:.4f}")

converted                0.0   1.0
acquisition_channel               
email                61468  9219
organic              49679  7530
paid_search          59114  8491
referral             51813  7729
social               52189  8015

Chi-square: 18.2650 (dof=4)
p-value:    0.0011

- **Result:** $\chi^2 = 18.2650$ ($\text{dof} = 4$), $p = 0.0011$ (after standardizing channel casing into 5 clean categories: `email` `13.04%`, `organic` `13.16%`, `paid_search` `12.56%`, `referral` `12.98%`, `social` `13.31%`)
- After cleaning inconsistent capitalization (`Email`/`email` and `paid_Search`/`paid_search`), the Chi-square test shows a statistically significant association between acquisition channel and conversion ($p = 0.0011 < 0.05$). However, the actual conversion rates across channels are very close, ranging narrowly between `12.56%` (paid search) and `13.31%` (social).
- **Business Meaning:** While the large merged sample size detects a slight lag in `paid_search` conversion (`12.56%`) compared to `social` (`13.31%`) and `organic` (`13.16%`), all five channels convert near ~13%. Marketing teams should audit paid search landing-page alignment for incremental gains while focusing primarily on cost-per-acquisition efficiency.

### Q7. Is product category associated with discount usage?

In [8]:
cat_disc = tx_cust[["product_category", "discount_used"]].dropna()
cont_table = pd.crosstab(cat_disc["product_category"], cat_disc["discount_used"])

chi2, p_val, dof, expected = stats.chi2_contingency(cont_table)

print(cont_table)
print(f"\nChi-square: {chi2:.4f}")
print(f"p-value:    {p_val:.4f}")

discount_used        0     1
product_category            
beauty            5305  2281
electronics       5212  2212
fashion           5326  2255
home              5210  2278
sports            5225  2279
toys              5196  2221

Chi-square: 1.4622
p-value:    0.9174

- **Result:** $\chi^2 = 1.4622$ ($\text{dof} = 5$), $p = 0.9174$ (Discount usage is ~30% across every category)
- Because $p = 0.9174 > 0.05$, discount usage is completely independent of product category—customers apply discounts at virtually the exact same ~30% rate across all six categories.
- **Business Meaning:** Promotional codes are currently redeemed uniformly across the catalog. Category managers can safely run targeted category-specific promotions (e.g., clearing seasonal inventory) without worrying that one category already monopolizes discount usage.

### Q8. Is time on site correlated with purchase value?

In [9]:
time_val = tx_web[["time_on_site", "purchase_value"]].dropna()
corr, p_val = stats.pearsonr(time_val["time_on_site"], time_val["purchase_value"])

print(f"Correlation (r): {corr:.6f}")
print(f"p-value:         {p_val:.4f}")

Correlation (r): 0.000164
p-value:         0.9267

- **Result:** $r = 0.0002$, $p = 0.9267$
- There is zero linear correlation between time spent on the site and transaction purchase value ($r \approx 0$, $p = 0.9267 > 0.05$).
- **Business Meaning:** Longer browsing sessions do not translate into higher spending per order. UX improvements should prioritize friction-free product discovery rather than artificially extending dwell time.

### Q9. Is number of pages viewed correlated with purchase value?

In [10]:
pages_val = tx_web[["pages_viewed", "purchase_value"]].dropna()
corr, p_val = stats.pearsonr(pages_val["pages_viewed"], pages_val["purchase_value"])

print(f"Correlation (r): {corr:.6f}")
print(f"p-value:         {p_val:.4f}")

Correlation (r): -0.003584
p-value:         0.0442

- **Result:** $r = -0.0036$, $p = 0.0442$
- Although $p = 0.0442$ crosses the conventional $0.05$ significance threshold due to the very large sample size ($N = 313,313$ joined records), the correlation coefficient ($r = -0.0036$) is practically zero.
- **Business Meaning:** The number of pages viewed has no meaningful impact on how much a customer spends per order; merchandising strategies should not rely on increasing pageviews to raise order value.

### Q10. Is time on site correlated with pages viewed?

In [11]:
time_pages = web[["time_on_site", "pages_viewed"]].dropna()
corr, p_val = stats.pearsonr(time_pages["time_on_site"], time_pages["pages_viewed"])

print(f"Correlation (r): {corr:.6f}")
print(f"p-value:         {p_val:.4f}")

Correlation (r): 0.004540
p-value:         0.2293

- **Result:** $r = 0.0045$, $p = 0.2293$
- With $r = 0.0045$ and $p = 0.2293 > 0.05$, there is no statistically significant linear correlation between session duration and the number of pages viewed.
- **Business Meaning:** Users who spend more time on the site are not necessarily browsing more pages—some linger on a single product page while others rapidly click through multiple pages. Engagement tracking should monitor both metrics separately.

## 3. Analytical Challenges

### C1. Explore whether high-spending customers behave differently in terms of session time.
*Definition:* High-spending customers are defined as customers in the top quartile ($\ge 75\text{th}$ percentile) of cumulative lifetime purchase value.

In [12]:
# Aggregate total spend per customer first to avoid duplicating transaction amounts
cust_spend = transactions.groupby("customer_id")["purchase_value"].sum().reset_index()
cust_web = cust_spend.merge(web, on="customer_id", how="left")

threshold = cust_spend["purchase_value"].quantile(0.75)

high = cust_web[cust_web["purchase_value"] >= threshold]["time_on_site"].dropna()
low = cust_web[cust_web["purchase_value"] < threshold]["time_on_site"].dropna()

t_stat, p_val = stats.ttest_ind(high, low, equal_var=False)

print(f"75th percentile spend threshold: ${threshold:.2f}")
print(f"High-spending mean time:         {high.mean():.2f} sec")
print(f"Low-spending mean time:          {low.mean():.2f} sec")
print(f"t-statistic:                     {t_stat:.4f}")
print(f"p-value:                         {p_val:.4f}")

75th percentile spend threshold: $1338.19
High-spending mean time:         301.60 sec
Low-spending mean time:          300.64 sec
t-statistic:                     0.9224
p-value:                         0.3563

- **Result:** High-Spender Mean Time = `301.60s`, Other Customers Mean Time = `300.64s` | $t = 0.9224$, $p = 0.3563$
- Because $p = 0.3563 > 0.05$, top-quartile spenders spend virtually the exact same amount of time per session (~5 minutes) as lower-spending customers.
- **Business Meaning:** Session duration cannot be used as a behavioral proxy to identify high-value VIP customers. Predictive lead scoring should rely on purchase frequency and basket composition rather than time on site.

### C2. Identify a relationship that is meaningful for business strategy.

In [13]:
pages_conv = web[["pages_viewed", "converted"]].dropna()
corr, p_val = stats.pearsonr(pages_conv["pages_viewed"], pages_conv["converted"])

print(f"Correlation (r): {corr:.4f}")
print(f"p-value:         {p_val:.4e}")

Correlation (r): 0.1279
p-value:         1.5307e-253

- **Result:** $r = 0.1279$, $p = 1.53 \times 10^{-253}$
- There is a highly significant positive correlation between the number of pages viewed in a session and whether the session converts into a purchase ($r = 0.1279$, $p < 0.0001$).
- **Business Meaning:** Unlike raw session duration (which showed no effect), active browsing depth meaningfully correlates with conversion. Investing in personalized product recommendations, intuitive category filtering, and *"Frequently Bought Together"* carousels to encourage multi-page exploration is a high-leverage strategy to lift conversion rates.

### C3. Identify a statistically significant result that is not practically meaningful.

In [14]:
pages_val = tx_web[["pages_viewed", "purchase_value"]].dropna()
corr, p_val = stats.pearsonr(pages_val["pages_viewed"], pages_val["purchase_value"])

print(f"Correlation (r): {corr:.6f}")
print(f"p-value:         {p_val:.4f}")
print(f"Variance explained (R^2): {corr**2:.8f} ({corr**2 * 100:.6f}%)")

Correlation (r): -0.003584
p-value:         0.0442
Variance explained (R^2): 0.00001285 (0.001285%)

- **Result:** `pages_viewed` vs. `purchase_value` (from Q9): $r = -0.0036$, $p = 0.0442$, $R^2 = 0.000013$ (`0.0013%` of variance explained)
- Because $p = 0.0442 < 0.05$, the negative correlation between pages viewed and purchase value is technically **statistically significant** at the 5% level. However, a correlation of $r = -0.0036$ is indistinguishable from zero in real-world terms.
- **Business Meaning — Why Statistical Significance $\neq$ Business Importance:**  
  With a very large sample size ($N = 313,313$ merged rows), hypothesis tests have high statistical power to detect even microscopic deviations from zero ($p < 0.05$). Yet an $R^2$ of `0.0013%` means pageviews explain virtually none of the variation in order value. Acting on $p$-values alone without checking **effect size** ($r$, $R^2$, or mean differences) can lead teams to waste engineering and marketing resources chasing noise that produces zero commercial impact.

---

## 4. Conclusion

This statistical evaluation of the e-commerce platform reveals clear guidance on where to focus growth efforts—and where not to over-optimize:
- **What Drives Conversion:** Active browsing depth (`pages_viewed`, $r = 0.1279$, $p < 10^{-250}$) is a genuine positive predictor of session conversion, whereas device type ($p = 0.5626$) and raw session time ($p = 0.3563$) show no meaningful differentiation.
- **What Drives Order Value (AOV):** Average transaction value (~$235) is remarkably invariant across countries, genders, acquisition channels, discount usage, and browsing depth. Raising AOV requires structural basket-building incentives (bundles, tiered thresholds) rather than untargeted discounting.
- **Statistical Discipline:** Large datasets frequently yield low $p$-values for trivial effects (such as `pages_viewed` vs. `purchase_value`, $r = -0.0036$, $p = 0.0442$). Combining $p$-values with effect sizes and proper table granularity prevents false-positive business decisions.